# 최소 직접 N·V M2 — Dunnhumby seed49 예비 실행

M1과 M2만 각각 300epoch 학습합니다. ID 임베딩만 이진 LightGCN으로 전파하고, 사용자의 중심화된 $q_N,q_V$와 상품의 구매자 평균 $q_N$, 구매금액 백분위를 전파 뒤 결합합니다. 학습 파라미터는 $\gamma_N,\gamma_V$ 두 스칼라뿐이며 같은 BPR 손실·optimizer로 공동학습합니다.

- 학습: DAY 1–704, test: DAY 705–711, validation·holdout 없음
- 신규상품 추천, MIN_ITEM_INTER=1, binary graph, uniform negative K=1, 표본 가중 없음
- test는 고정 300epoch 마지막 checkpoint에서 모형당 한 번만 평가
- 마지막 주 test가 이미 노출된 뒤 설계한 모형이므로 결과는 post-hoc 탐색이며, 단일 seed로 최종 성능·유의성을 주장하지 않음
- test 결과를 보고 구조·가중치·epoch를 다시 조정하지 않음


## 1. Drive 연결·고정 소스 로드


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = 'b52f7aac7a7dec724bde8bc1fd99d3639d06e63d'
REPO = Path('/content/clv-m2-direct-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 설정·데이터 준비 — 여기서는 학습하지 않음


In [ ]:
import clv_m2_direct_nv_lastweek as experiment
cfg = experiment.configure()
cfg, prepared = experiment.prepare(cfg)
assert cfg.seeds == (49,) and cfg.epochs == 300
assert prepared['protocol']['pilot_only'] and prepared['protocol']['previously_exposed_test']
print('자동재개 경로:', prepared['run_dir'])


## 3. M1·M2 순차 학습 및 마지막 checkpoint 평가


In [ ]:
result = experiment.run(cfg, prepared)
assert result['reading']['complete'] and result['reading']['seed_count'] == 1
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 전체 원본 결과 확인


In [ ]:
from IPython.display import display
display(result['absolute'])
display(result['paired_comparison'])
display(result['diagnostics'])
display(result['summary'])
print(json.dumps(result['paths'], ensure_ascii=False, indent=2))


## 5. 원본 결과 ZIP 다운로드


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content/m2_direct_nv_lastweek_seed49_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
    archive.write(prepared['run_dir'] / 'protocol.json', arcname='protocol.json')
files.download(str(zip_path))
